## Synthetically generate humanised ironic evaluation dataset

To run, activate the `openai_environment` conda environment specified in the environements directory. Furthermore, replace the API_KEY placeholder with a real OpenAI API key.

In [ ]:
from openai import OpenAI
import json

# instantiate api connection
client = OpenAI(
  api_key="API-KEY"
)

output_total = 300
batch_size = 10
iteration_total = output_total//batch_size

# tracks the comments that have been 
# previously generated
seen = set()
dataset = []

# helper function to generate a batch
# of comments 
def generate_comments(past_comments):
    last_50_comments = past_comments[-50:]
    raw_output = client.responses.create(
        model="gpt-5.2",
        input = [
            {
                "role": "system",
                "content": ( "You are a data generator. You generate synthetic feedback comments. Return only valid JSON with no extra text. Each object schema: [{\"comment\": \"...\", \"label\": 0 or 1}, {\"comment\": \"...\", \"label\": 0 or 1}, ...]. All generated comments must be UNIQUE." 
                )
            },
            {
                "role": "user",
                "content": (
                    f"Generate exactly {batch_size} unique student comments and corresponding labels of whether they are ironic. 0 = not ironic and 1 = ironic. There should be equal number of ironic and not ironic comments. At least 25% of comments should have 2-3 sentences. Comments MUST have spelling mistakes, grammar errors and abbreviations. Do not repeat comments already in this list: " + 
                    "\n".join([f"- {comment}" for comment in last_50_comments])
                )
            }
        ],
        store=True,
    ).output_text
    
    processed_json = json.loads(raw_output)
    return processed_json

# generate all batches of comments
for i in range(iteration_total):
    past_comments = [entry["comment"] for entry in dataset]
    new_comments = generate_comments(past_comments)

    new_entries = []
    for new_comment in new_comments:
        c = new_comment["comment"]
        if c not in seen:
            seen.add(c)
            new_entries.append(new_comment)

    dataset.extend(new_entries)


# save comments produced
with open("dataset/test.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)


[
  {
    "comment": "I appreciated the group activity; it made the concept click. Thanks for the clear examples.",
    "label": 0
  },
  {
    "comment": "Could you please post the solutions from last week's quiz?",
    "label": 0
  },
  {
    "comment": "The pacing was good today, and the slides were helpful.",
    "label": 0
  },
  {
    "comment": "I struggled with question 3, but the tutorial clarified it. I'll review the notes again.",
    "label": 0
  },
  {
    "comment": "Great lecture! The analogy with the bridge was memorable.",
    "label": 0
  },
  {
    "comment": "This assignment is a joke and whoever wrote it has no clue.",
    "label": 1
  },
  {
    "comment": "Your feedback was useless. Stop wasting our time.",
    "label": 1
  },
  {
    "comment": "I busted my butt on this and the grading still sucked.",
    "label": 1
  },
  {
    "comment": "Dude, your group presentation was trash, and you talked in circles.",
    "label": 1
  },
  {
    "comment": "Quit giving t